# Evaluate Starlar Fine-Tuned LLM

This notebook evaluates the Starlar v2 fine-tuned Mistral QLoRA model.

Main comparison:
- Base Mistral-7B-Instruct-v0.2
- Starlar v2 fine-tuned Mistral QLoRA

Evaluation setup:
1. Controlled Starlar test set evaluation using gold/source-grounded prompts.
2. Optional: test the fine-tuned generator on the old best RAG retrieved contexts.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU yok. Runtime > Change runtime type > L4 veya T4 GPU seç.")

CUDA available: True
GPU: NVIDIA L4


In [3]:
import os

project_path = "/content/drive/MyDrive/turkish_legal_rag"

processed_path = f"{project_path}/data/processed"
outputs_path = f"{project_path}/outputs"
metrics_path = f"{outputs_path}/metrics"
models_path = f"{outputs_path}/models"

starlar_test_path = f"{processed_path}/starlar_llm_sft_v2_test.jsonl"

adapter_path = f"{models_path}/mistral_legal_qlora_starlar_v2_800steps"

print("Starlar test exists:", os.path.exists(starlar_test_path))
print("Adapter folder exists:", os.path.exists(adapter_path))
print("Adapter config exists:", os.path.exists(f'{adapter_path}/adapter_config.json'))
print("Adapter model exists:", os.path.exists(f'{adapter_path}/adapter_model.safetensors'))

Starlar test exists: True
Adapter folder exists: True
Adapter config exists: True
Adapter model exists: True


In [4]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 144.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.5 MB/s eta 0:00:00


In [5]:
import os
import gc
import json
import re
import difflib
import pandas as pd
import numpy as np
import torch

from tqdm import tqdm
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel

In [6]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
dataset = load_dataset(
    "json",
    data_files={"test": starlar_test_path}
)

test_df = pd.DataFrame(dataset["test"])

print("Full test shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

eval_df = test_df.sample(n=20, random_state=42).reset_index(drop=True)

print("Eval sample:", eval_df.shape)

display(eval_df[[
    "id",
    "user_content",
    "assistant_content",
    "metadata_source",
    "metadata_variant_type"
]].head())

Generating test split: 0 examples [00:00, ? examples/s]

Full test shape: (400, 8)
Columns: ['id', 'text', 'system_content', 'user_content', 'assistant_content', 'metadata_source', 'metadata_variant_type', 'metadata_source_id']
Eval sample: (20, 8)


,id,user_content,assistant_content,metadata_source,metadata_variant_type
0,sft_exp_006056,[Kaynak]\nBaşlık: Medeni Hukuk - Miras/Aile/Eş...,Özet: Vasiyetnamenin geçerli olabilmesi için e...,ORICON,short_summary_with_citation
1,sft_exp_004303,[Kaynak]\nBaşlık: Genel hukuk / sınıflandırma ...,Kaynağa göre: Dijital sınavların güvenliğini s...,ORICON,direct_grounded_answer
2,sft_exp_010876,[Kaynak]\nBaşlık: İdare ve İdari Yargılama Huk...,Kaynağa göre: Acele kamulaştırma kararından so...,ORICON,direct_grounded_answer
3,sft_exp_009682,[Kaynak]\nBaşlık: İcra ve İflas Hukuku\nKaynak...,Kaynağa göre: İşvereninden alacaklarını alamay...,ORICON,direct_grounded_answer
4,sft_exp_004845,[Kaynak]\nBaşlık: Genel hukuk / sınıflandırma ...,Bu açıklama şu kaynak bilgilerine dayandırılma...,ORICON,source_limited_explanation


In [8]:
RESPONSE_MARKER = "[/INST]"

def extract_prompt_from_sft_text(text):
    text = str(text)

    if RESPONSE_MARKER in text:
        return text.split(RESPONSE_MARKER)[0] + RESPONSE_MARKER

    return text


eval_df["prompt"] = eval_df["text"].apply(extract_prompt_from_sft_text)

print("PROMPT SAMPLE:")
print(eval_df.loc[0, "prompt"][-1800:])

print("\nEXPECTED ANSWER:")
print(eval_df.loc[0, "assistant_content"])

PROMPT SAMPLE:
<s>[INST] Sen bir Türk hukuku RAG asistanısın. Yalnızca kullanıcı tarafından verilen kaynak metne dayanarak cevap ver. Kaynakta olmayan bilgiyi üretme. Cevabın sonunda kaynak/citation bilgisini mutlaka belirt. Kaynak kesin resmî karar metni değilse bunu kesin hüküm gibi sunma.

[Kaynak]
Başlık: Medeni Hukuk - Miras/Aile/Eşya/Kişiler
Kaynak: ORICON
Dosya: ORICON_clean_legal_source_for_chunking.txt
Chunk ID: oricon_medeni_000088
Citation: ORICON - Medeni Hukuk - Miras/Aile/Eşya/Kişiler - oricon_medeni_000088
Metin: Vasiyetnamenin geçerli olabilmesi için en temel şart, vasiyet edenin ayırt etme gücüne sahip olmasıdır.

Soru: Bu kaynak parçasını kısa ve kaynaklı şekilde özetle. [/INST]

EXPECTED ANSWER:
Özet: Vasiyetnamenin geçerli olabilmesi için en temel şart, vasiyet edenin ayırt etme gücüne sahip olmasıdır.

Kaynak: ORICON - Medeni Hukuk - Miras/Aile/Eşya/Kişiler - oricon_medeni_000088


In [9]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
tokenizer.truncation_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Tokenizer ready.")
print("Padding side:", tokenizer.padding_side)
print("Truncation side:", tokenizer.truncation_side)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Tokenizer ready.
Padding side: right
Truncation side: left


In [10]:
MAX_INPUT_LENGTH = 1536

def generate_answer_only(model, tokenizer, prompt, max_new_tokens=180, max_length=MAX_INPUT_LENGTH):
    tokenizer.truncation_side = "left"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip()


def clean_answer(text):
    text = str(text).strip()

    unwanted_markers = [
        "Cevap:",
        "Yanıt:",
        "Answer:"
    ]

    for marker in unwanted_markers:
        if marker in text:
            text = text.split(marker)[-1].strip()

    return text

In [11]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def token_f1(prediction, reference):
    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    common = {}

    for t in pred_tokens:
        common[t] = common.get(t, 0) + 1

    overlap = 0

    for t in ref_tokens:
        if common.get(t, 0) > 0:
            overlap += 1
            common[t] -= 1

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)


def text_similarity(prediction, reference):
    return difflib.SequenceMatcher(
        None,
        normalize_text(prediction),
        normalize_text(reference)
    ).ratio()


def has_source_citation(answer):
    answer = str(answer).lower()
    return int("kaynak:" in answer or "citation:" in answer)

Part A — Base Mistral generation

In [12]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

base_model.eval()

print("Base Mistral loaded.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Base Mistral loaded.


In [13]:
base_answers = []

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    answer = generate_answer_only(
        model=base_model,
        tokenizer=tokenizer,
        prompt=row["prompt"],
        max_new_tokens=180,
        max_length=MAX_INPUT_LENGTH
    )

    base_answers.append(clean_answer(answer))

eval_df["base_generated_answer"] = base_answers

display(eval_df[[
    "id",
    "assistant_content",
    "base_generated_answer"
]].head())

  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 20/20 [03:27<00:00, 10.38s/it]


,id,assistant_content,base_generated_answer
0,sft_exp_006056,Özet: Vasiyetnamenin geçerli olabilmesi için e...,The validity of a will requires the most funda...
1,sft_exp_004303,Kaynağa göre: Dijital sınavların güvenliğini s...,"According to the given source from ORICON, ens..."
2,sft_exp_010876,Kaynağa göre: Acele kamulaştırma kararından so...,"According to the given source from ORICON, the..."
3,sft_exp_009682,Kaynağa göre: İşvereninden alacaklarını alamay...,According to the given source from ORICON unde...
4,sft_exp_004845,Bu açıklama şu kaynak bilgilerine dayandırılma...,"The term ""Hak düşürücü sürelerin yeniden canla..."


In [14]:
base_intermediate_path = f"{metrics_path}/controlled_starlar_base_generation_intermediate_20.csv"

eval_df.to_csv(
    base_intermediate_path,
    index=False,
    encoding="utf-8-sig"
)

print("Base intermediate saved:", base_intermediate_path)

Base intermediate saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_generation_intermediate_20.csv


In [15]:
del base_model

gc.collect()
torch.cuda.empty_cache()

print("Base model deleted from memory.")

Base model deleted from memory.


Part B — Starlar fine-tuned model generation

In [16]:
ft_base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

ft_base_model.eval()

ft_model = PeftModel.from_pretrained(
    ft_base_model,
    adapter_path
)

ft_model.eval()

print("Starlar fine-tuned Mistral loaded.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Starlar fine-tuned Mistral loaded.


In [22]:
finetuned_answers = []

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    answer = generate_answer_only(
        model=ft_model,
        tokenizer=tokenizer,
        prompt=row["prompt"],
        max_new_tokens=260,
        max_length=MAX_INPUT_LENGTH
    )

    finetuned_answers.append(clean_answer(answer))

eval_df["finetuned_generated_answer"] = finetuned_answers

display(eval_df[[
    "id",
    "assistant_content",
    "base_generated_answer",
    "finetuned_generated_answer"
]].head())

  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 20/20 [09:21<00:00, 28.09s/it]


,id,assistant_content,base_generated_answer,finetuned_generated_answer
0,sft_exp_006056,Özet: Vasiyetnamenin geçerli olabilmesi için e...,The validity of a will requires the most funda...,Özet: Vasiyetnamenin geçerli olabilmesi için e...
1,sft_exp_004303,Kaynağa göre: Dijital sınavların güvenliğini s...,"According to the given source from ORICON, ens...",Kaynağa göre: Dijital sınavların güvenliğini s...
2,sft_exp_010876,Kaynağa göre: Acele kamulaştırma kararından so...,"According to the given source from ORICON, the...",Kaynağa göre: Acele kamulaştırma kararından so...
3,sft_exp_009682,Kaynağa göre: İşvereninden alacaklarını alamay...,According to the given source from ORICON unde...,Kaynağa göre: İşvereninden alacaklarını alamay...
4,sft_exp_004845,Bu açıklama şu kaynak bilgilerine dayandırılma...,"The term ""Hak düşürücü sürelerin yeniden canla...",Bu açıklama şu kaynak bilgilerine dayandırılma...


In [23]:
eval_df["base_token_f1"] = eval_df.apply(
    lambda row: token_f1(row["base_generated_answer"], row["assistant_content"]),
    axis=1
)

eval_df["finetuned_token_f1"] = eval_df.apply(
    lambda row: token_f1(row["finetuned_generated_answer"], row["assistant_content"]),
    axis=1
)

eval_df["base_similarity"] = eval_df.apply(
    lambda row: text_similarity(row["base_generated_answer"], row["assistant_content"]),
    axis=1
)

eval_df["finetuned_similarity"] = eval_df.apply(
    lambda row: text_similarity(row["finetuned_generated_answer"], row["assistant_content"]),
    axis=1
)

eval_df["base_has_source"] = eval_df["base_generated_answer"].apply(has_source_citation)
eval_df["finetuned_has_source"] = eval_df["finetuned_generated_answer"].apply(has_source_citation)

auto_summary_df = pd.DataFrame([
    {
        "method": "Base Mistral",
        "mean_token_f1": eval_df["base_token_f1"].mean(),
        "mean_text_similarity": eval_df["base_similarity"].mean(),
        "source_citation_rate": eval_df["base_has_source"].mean()
    },
    {
        "method": "Starlar Fine-tuned Mistral QLoRA",
        "mean_token_f1": eval_df["finetuned_token_f1"].mean(),
        "mean_text_similarity": eval_df["finetuned_similarity"].mean(),
        "source_citation_rate": eval_df["finetuned_has_source"].mean()
    }
])

display(auto_summary_df)

display(eval_df[[
    "id",
    "base_token_f1",
    "finetuned_token_f1",
    "base_similarity",
    "finetuned_similarity",
    "base_has_source",
    "finetuned_has_source"
]])

,method,mean_token_f1,mean_text_similarity,source_citation_rate
0,Base Mistral,0.184815,0.155437,0.25
1,Starlar Fine-tuned Mistral QLoRA,0.965736,0.969455,0.85


,id,base_token_f1,finetuned_token_f1,base_similarity,finetuned_similarity,base_has_source,finetuned_has_source
0,sft_exp_006056,0.327869,1.000000,0.467337,1.000000,0,1
1,sft_exp_004303,0.096154,1.000000,0.065574,1.000000,1,1
2,sft_exp_010876,0.011976,0.985714,0.024955,0.998016,0,1
3,sft_exp_009682,0.250000,1.000000,0.108772,1.000000,0,1
4,sft_exp_004845,0.156863,1.000000,0.194891,1.000000,0,1
5,sft_exp_010345,0.289157,1.000000,0.267961,1.000000,0,1
6,sft_exp_016869,0.063492,0.735751,0.046395,0.739098,0,0
7,sft_exp_002778,0.123711,1.000000,0.022567,1.000000,0,1
8,sft_exp_006111,0.289474,1.000000,0.257198,1.000000,0,1
9,sft_exp_010801,0.093750,1.000000,0.039900,1.000000,0,1


In [24]:
controlled_results_path = f"{metrics_path}/controlled_starlar_base_vs_finetuned_generation_results_20.csv"
controlled_auto_summary_path = f"{metrics_path}/controlled_starlar_base_vs_finetuned_auto_summary_20.csv"

eval_df.to_csv(
    controlled_results_path,
    index=False,
    encoding="utf-8-sig"
)

auto_summary_df.to_csv(
    controlled_auto_summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("Controlled results saved:", controlled_results_path)
print("Controlled auto summary saved:", controlled_auto_summary_path)

Controlled results saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_generation_results_20.csv
Controlled auto summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_auto_summary_20.csv


In [25]:
manual_scoring_df = eval_df.copy()

manual_scoring_df["base_manual_score"] = ""
manual_scoring_df["finetuned_manual_score"] = ""
manual_scoring_df["base_error_type"] = ""
manual_scoring_df["finetuned_error_type"] = ""
manual_scoring_df["notes"] = ""

manual_scoring_path = f"{metrics_path}/controlled_starlar_base_vs_finetuned_manual_scoring_template_20.csv"

manual_scoring_df.to_csv(
    manual_scoring_path,
    index=False,
    encoding="utf-8-sig"
)

print("Manual scoring template saved:", manual_scoring_path)

Manual scoring template saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_manual_scoring_template_20.csv


In [26]:
for i, row in eval_df.head(10).iterrows():
    print("=" * 120)
    print("INDEX:", i)
    print("ID:", row["id"])

    print("\nUSER PROMPT LAST PART:")
    print(row["prompt"][-1200:])

    print("\nEXPECTED:")
    print(row["assistant_content"])

    print("\nBASE:")
    print(row["base_generated_answer"])

    print("\nFINE-TUNED:")
    print(row["finetuned_generated_answer"])

    print("\nMETRICS:")
    print("base_token_f1:", row["base_token_f1"])
    print("finetuned_token_f1:", row["finetuned_token_f1"])
    print("base_similarity:", row["base_similarity"])
    print("finetuned_similarity:", row["finetuned_similarity"])
    print("base_has_source:", row["base_has_source"])
    print("finetuned_has_source:", row["finetuned_has_source"])

INDEX: 0
ID: sft_exp_006056

USER PROMPT LAST PART:
<s>[INST] Sen bir Türk hukuku RAG asistanısın. Yalnızca kullanıcı tarafından verilen kaynak metne dayanarak cevap ver. Kaynakta olmayan bilgiyi üretme. Cevabın sonunda kaynak/citation bilgisini mutlaka belirt. Kaynak kesin resmî karar metni değilse bunu kesin hüküm gibi sunma.

[Kaynak]
Başlık: Medeni Hukuk - Miras/Aile/Eşya/Kişiler
Kaynak: ORICON
Dosya: ORICON_clean_legal_source_for_chunking.txt
Chunk ID: oricon_medeni_000088
Citation: ORICON - Medeni Hukuk - Miras/Aile/Eşya/Kişiler - oricon_medeni_000088
Metin: Vasiyetnamenin geçerli olabilmesi için en temel şart, vasiyet edenin ayırt etme gücüne sahip olmasıdır.

Soru: Bu kaynak parçasını kısa ve kaynaklı şekilde özetle. [/INST]

EXPECTED:
Özet: Vasiyetnamenin geçerli olabilmesi için en temel şart, vasiyet edenin ayırt etme gücüne sahip olmasıdır.

Kaynak: ORICON - Medeni Hukuk - Miras/Aile/Eşya/Kişiler - oricon_medeni_000088

BASE:
The validity of a will requires the most fundamen

In [27]:
eval_summary_df = pd.DataFrame([{
    "experiment": "Controlled Starlar gold-context evaluation",
    "base_model": model_name,
    "fine_tuned_adapter": adapter_path,
    "eval_sample_size": len(eval_df),
    "max_input_length": MAX_INPUT_LENGTH,
    "max_new_tokens": 180,
    "results_path": controlled_results_path,
    "auto_summary_path": controlled_auto_summary_path,
    "manual_scoring_template_path": manual_scoring_path
}])

controlled_eval_summary_path = f"{metrics_path}/controlled_starlar_base_vs_finetuned_eval_summary_20.csv"

eval_summary_df.to_csv(
    controlled_eval_summary_path,
    index=False,
    encoding="utf-8-sig"
)

display(eval_summary_df)

print("Controlled eval summary saved:", controlled_eval_summary_path)

,experiment,base_model,fine_tuned_adapter,eval_sample_size,max_input_length,max_new_tokens,results_path,auto_summary_path,manual_scoring_template_path
0,Controlled Starlar gold-context evaluation,mistralai/Mistral-7B-Instruct-v0.2,/content/drive/MyDrive/turkish_legal_rag/outpu...,20,1536,180,/content/drive/MyDrive/turkish_legal_rag/outpu...,/content/drive/MyDrive/turkish_legal_rag/outpu...,/content/drive/MyDrive/turkish_legal_rag/outpu...


Controlled eval summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_eval_summary_20.csv


In [28]:
files_to_check = [
    controlled_results_path,
    controlled_auto_summary_path,
    manual_scoring_path,
    controlled_eval_summary_path
]

print("FINAL CHECK")
print("=" * 80)

for file in files_to_check:
    print(file)
    print("Exists:", os.path.exists(file))
    if os.path.exists(file):
        print("Size KB:", round(os.path.getsize(file) / 1024, 2))
    print("-" * 80)

print("Done.")

FINAL CHECK
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_generation_results_20.csv
Exists: True
Size KB: 93.61
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_auto_summary_20.csv
Exists: True
Size KB: 0.19
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_manual_scoring_template_20.csv
Exists: True
Size KB: 93.79
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_eval_summary_20.csv
Exists: True
Size KB: 0.68
--------------------------------------------------------------------------------
Done.


In [29]:
import os
import numpy as np
import pandas as pd

# Eğer eval_df hâlâ bellekte varsa onu kullanır.
# Eğer runtime değiştiyse results CSV'den tekrar yükler.
controlled_results_path = f"{metrics_path}/controlled_starlar_base_vs_finetuned_generation_results_20.csv"

if "eval_df" in globals():
    controlled_scored_df = eval_df.copy()
else:
    controlled_scored_df = pd.read_csv(controlled_results_path)

print("Loaded rows:", len(controlled_scored_df))

if len(controlled_scored_df) != 20:
    raise ValueError(f"Expected 20 rows, but got {len(controlled_scored_df)} rows.")

base_manual_scores = [
    0.5,  # 0 - partly correct but English and meaning slightly distorted
    0.5,  # 1 - mostly correct but English and citation format differs
    0.5,  # 2 - partially captures answer but legal terms are mistranslated
    0.5,  # 3 - partially correct but English and "conciliation" is not ideal
    0.0,  # 4 - hallucinated intellectual property explanation
    0.5,  # 5 - partially correct but says Constitution instead of Danıştay Law
    0.0,  # 6 - does not follow expected source-grounded template
    0.0,  # 7 - broken/incomplete answer
    0.5,  # 8 - semantically close but English and format mismatch
    0.5,  # 9 - partially correct but misses "acele kamulaştırma"
    0.5,  # 10 - partially related but contains legal mistranslations
    0.5,  # 11 - semantically close but English/format mismatch
    0.5,  # 12 - partially correct but broken Turkish and incomplete citation
    0.5,  # 13 - semantically close but English/format mismatch
    0.0,  # 14 - wrong explanation about stolen items
    0.5,  # 15 - partially captures seller non-performance but inaccurate
    0.5,  # 16 - partially correct but legal meaning is distorted
    0.5,  # 17 - semantically close but English/format mismatch
    0.5,  # 18 - semantically close but English/format mismatch
    0.5   # 19 - semantically close but English/format mismatch
]

finetuned_manual_scores = [
    1.0,  # 0
    1.0,  # 1
    1.0,  # 2 - content is correct; citation ID slightly truncated but source is clear
    1.0,  # 3
    1.0,  # 4
    1.0,  # 5
    0.5,  # 6 - strong partial answer but long output is truncated
    1.0,  # 7
    1.0,  # 8
    1.0,  # 9
    0.5,  # 10 - correct start but long output is truncated before full citation
    1.0,  # 11
    1.0,  # 12
    1.0,  # 13
    1.0,  # 14
    0.5,  # 15 - correct start but long output is truncated
    1.0,  # 16
    1.0,  # 17
    1.0,  # 18
    1.0   # 19
]

base_error_types = [
    "partial_english_answer",
    "partial_english_answer",
    "partial_mistranslation",
    "partial_english_answer",
    "hallucination",
    "partial_wrong_legal_source",
    "template_mismatch",
    "broken_incomplete_answer",
    "partial_english_answer",
    "partial_answer",
    "partial_mistranslation",
    "partial_english_answer",
    "partial_broken_answer",
    "partial_english_answer",
    "hallucination",
    "partial_mistranslation",
    "partial_distorted_legal_meaning",
    "partial_english_answer",
    "partial_english_answer",
    "partial_english_answer"
]

finetuned_error_types = [
    "correct",
    "correct",
    "correct_minor_citation_truncation",
    "correct",
    "correct",
    "correct",
    "partial_truncated_long_answer",
    "correct",
    "correct",
    "correct",
    "partial_truncated_long_answer",
    "correct",
    "correct",
    "correct",
    "correct",
    "partial_truncated_long_answer",
    "correct",
    "correct",
    "correct",
    "correct"
]

notes = [
    "Fine-tuned answer matches expected format and content.",
    "Fine-tuned answer matches expected format and content.",
    "Fine-tuned answer captures the full legal content; citation ID is slightly truncated.",
    "Fine-tuned answer matches expected format and content.",
    "Base hallucinates an intellectual property explanation; fine-tuned is exact.",
    "Base confuses legal source; fine-tuned is exact.",
    "The expected answer is very long; fine-tuned answer is much closer but truncated.",
    "Fine-tuned answer matches expected format and content.",
    "Fine-tuned answer matches expected format and content.",
    "Fine-tuned answer matches expected format and content.",
    "The expected answer is very long; fine-tuned answer is strong but truncated.",
    "Fine-tuned answer matches expected format and content.",
    "Fine-tuned answer matches expected format and content.",
    "Fine-tuned answer matches expected format and content.",
    "Fine-tuned answer matches expected format and content.",
    "The expected answer is very long; fine-tuned answer is strong but truncated.",
    "Fine-tuned answer matches expected format and content.",
    "Fine-tuned answer matches expected format and content.",
    "Fine-tuned answer matches expected format and content.",
    "Fine-tuned answer matches expected format and content."
]

controlled_scored_df["base_manual_score"] = base_manual_scores
controlled_scored_df["finetuned_manual_score"] = finetuned_manual_scores
controlled_scored_df["base_error_type"] = base_error_types
controlled_scored_df["finetuned_error_type"] = finetuned_error_types
controlled_scored_df["manual_notes"] = notes

base_manual_accuracy = controlled_scored_df["base_manual_score"].mean()
finetuned_manual_accuracy = controlled_scored_df["finetuned_manual_score"].mean()

manual_summary_df = pd.DataFrame([
    {
        "method": "Base Mistral",
        "manual_accuracy": base_manual_accuracy,
        "mean_token_f1": controlled_scored_df["base_token_f1"].mean(),
        "mean_text_similarity": controlled_scored_df["base_similarity"].mean(),
        "source_citation_rate": controlled_scored_df["base_has_source"].mean(),
        "sample_count": len(controlled_scored_df)
    },
    {
        "method": "Starlar Fine-tuned Mistral QLoRA",
        "manual_accuracy": finetuned_manual_accuracy,
        "mean_token_f1": controlled_scored_df["finetuned_token_f1"].mean(),
        "mean_text_similarity": controlled_scored_df["finetuned_similarity"].mean(),
        "source_citation_rate": controlled_scored_df["finetuned_has_source"].mean(),
        "sample_count": len(controlled_scored_df)
    }
])

display(manual_summary_df)

manual_scored_path = f"{metrics_path}/controlled_starlar_base_vs_finetuned_manual_scored_20.csv"
manual_summary_path = f"{metrics_path}/controlled_starlar_base_vs_finetuned_manual_score_summary_20.csv"

controlled_scored_df.to_csv(
    manual_scored_path,
    index=False,
    encoding="utf-8-sig"
)

manual_summary_df.to_csv(
    manual_summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("Base manual accuracy:", base_manual_accuracy)
print("Fine-tuned manual accuracy:", finetuned_manual_accuracy)

print("Manual scored results saved:", manual_scored_path)
print("Manual summary saved:", manual_summary_path)

Loaded rows: 20


,method,manual_accuracy,mean_token_f1,mean_text_similarity,source_citation_rate,sample_count
0,Base Mistral,0.400,0.184815,0.155437,0.25,20
1,Starlar Fine-tuned Mistral QLoRA,0.925,0.965736,0.969455,0.85,20


Base manual accuracy: 0.4
Fine-tuned manual accuracy: 0.925
Manual scored results saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_manual_scored_20.csv
Manual summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_manual_score_summary_20.csv


In [30]:
files_to_check = [
    f"{metrics_path}/controlled_starlar_base_vs_finetuned_generation_results_20.csv",
    f"{metrics_path}/controlled_starlar_base_vs_finetuned_auto_summary_20.csv",
    f"{metrics_path}/controlled_starlar_base_vs_finetuned_eval_summary_20.csv",
    f"{metrics_path}/controlled_starlar_base_vs_finetuned_manual_scoring_template_20.csv",
    f"{metrics_path}/controlled_starlar_base_vs_finetuned_manual_scored_20.csv",
    f"{metrics_path}/controlled_starlar_base_vs_finetuned_manual_score_summary_20.csv"
]

print("FINAL CHECK WITH MANUAL SCORES")
print("=" * 80)

for file in files_to_check:
    print(file)
    print("Exists:", os.path.exists(file))
    if os.path.exists(file):
        print("Size KB:", round(os.path.getsize(file) / 1024, 2))
    print("-" * 80)

print("Done.")

FINAL CHECK WITH MANUAL SCORES
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_generation_results_20.csv
Exists: True
Size KB: 93.61
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_auto_summary_20.csv
Exists: True
Size KB: 0.19
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_eval_summary_20.csv
Exists: True
Size KB: 0.68
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/controlled_starlar_base_vs_finetuned_manual_scoring_template_20.csv
Exists: True
Size KB: 93.79
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/contr